# Experiment 1: adaptation behavior — PD

The main descriptive sweep has 256 trials per task: four bases × four learning rates × two L2-SP values × two adaptation modes × four dataset partitions. The main grid uses one sampled context/query batch per table visit and training seed 42.

Read coverage first. Then follow fixed-update monitoring, inspect endpoint factor effects and dataset heterogeneity, and finish with transfer and computing diagnostics. Monitoring uses a small fixed panel and ensemble; final held-out benchmark results belong in the separate results notebook. No winning recipe is selected here.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment1/01_training_pd')
report = cp.NotebookReport('Experiment 1: adaptation behavior — PD')
TRACK = 'pd'


## 1. Coverage before effects

All configured identities count in the denominator. Recipe maps are split by base/adaptation, not flattened into a 256-row chart.

In [ ]:
run = cp.load_campaign(1, TRACK)
cp.show(sink, cp.plot_coverage(run))
cp.show(sink, cp.plot_coverage_grid(run))
report.add('1. Trial coverage', cp.coverage_summary(run))

## 2. Fixed-update learning trajectories

Each plot fixes base, adaptation and sampling. L2-SP has separate panels and each panel shows at most four learning rates. A mean is left missing when an update-zero observation is absent at that milestone.

In [ ]:
monitor = cp.effects(run)
cp.show(sink, cp.plot_trajectory_pages(run))
report.add('2. Fixed-update trajectories', cp.effect_summary(monitor))

## 3. Endpoint response surfaces

These are effects at the configured endpoint, not the best observed milestone. Read them together with coverage; a partially complete recipe is not comparable to a fully complete recipe without qualification.

In [ ]:
endpoint = cp.endpoint_effects(run)
cp.show(sink, cp.plot_response_surfaces(endpoint, run.metric))
report.add('3. Endpoint responses', cp.effect_summary(endpoint))

## 4. Matched knob comparisons

Compare λ=0.003 with λ=0 while keeping learning rate, adaptation and dataset fixed. Separately compare frozen with full updates at matching settings. Positive contrast favors the named alternative, not necessarily a positive pretraining effect.

In [ ]:
anchor = cp.factor_contrasts(endpoint, 'l2sp_lambda', 0.0, 0.003)
adaptation = cp.factor_contrasts(endpoint, 'frozen', False, True)
cp.show(sink, cp.plot_contrasts(anchor, run.metric, 'L2-SP 0.003 minus 0'))
cp.show(sink, cp.plot_contrasts(adaptation, run.metric, 'Frozen minus full'))
report.add('4. Matched knob comparisons', 'L2-SP:\n'+cp.effect_summary(anchor,'contrast')+'\nAdaptation:\n'+cp.effect_summary(adaptation,'contrast'))

## 5. Which datasets move together?

Dataset heatmaps preserve every recipe and every dataset, with at most 12 rows per page. Within each fixed base/adaptation/sampling combination, pages share the same color scale.

In [ ]:
cp.show(sink, cp.plot_dataset_pages(endpoint, run.metric))
report.add('5. Dataset heterogeneity', endpoint.groupby('dataset').effect.agg(['count','median','min','max']).to_string() if not endpoint.empty else 'No endpoint effects.')

## 6. Seen versus held-out behavior

Changes are measured relative to each table’s own starting score. Training and held-out tables are different, so the comparison is a transfer diagnostic; it does not measure forgetting on non-credit data.

In [ ]:
cp.show(sink, cp.plot_train_test(run))
report.add('6. Seen versus held-out behavior', 'Training tables:\n'+cp.effect_summary(cp.endpoint_effects(run,'train'))+'\nHeld-out tables:\n'+cp.effect_summary(endpoint))

## 7. Cost and numerical health

Follow training loss and gradient clipping through bounded update windows, then inspect weight drift, time, memory and skipped updates alongside effects. Epoch diagnostics average records within trial/window before summarizing across trials; they are not extra independent dataset observations. A trial with a poor score remains part of this descriptive experiment; ordinary poor performance is not a numerical failure.

In [ ]:
cp.show(sink, cp.plot_optimization(run, 'train_loss'))
cp.show(sink, cp.plot_optimization(run, 'clipped_frac'))
cp.show(sink, cp.plot_diagnostics(run))
failures = display_frame(run.trials[run.trials.status.isin(['FAIL','DIVERGED','INTERRUPTED'])])
display(failures.drop(columns=['final_ckpt_path','source_file','train_dataset_ids','test_dataset_ids'],errors='ignore'))
report.add('7. Cost and numerical health', cp.coverage_summary(run)+'\n'+str(len(failures))+' failed/diverged/interrupted identities.')

## 8. Non-credit retention

Fixed public non-credit datasets remain outside the adaptation corpus. Curves compare every milestone with the same starting checkpoint, rows and monitoring seed. This small panel measures retention on those tables; it does not establish universal absence of forgetting.

In [ ]:
from src.visualize import diagnostics as dg
for current in [run]:
    cp.show(sink, cp.plot_trajectory_pages(current, split='ood'))
report.add('8. Non-credit retention', '\n\n'.join(c.track.upper()+'\n'+cp.effect_summary(cp.endpoint_effects(c, 'ood')) for c in [run]))

## 9. Parameter movement and sampled resources

Milestone tensor summaries complement total norm-weighted drift. Resource plots use one median per trial and omit unavailable counters; sampled device utilization is not precise kernel time or energy.

In [ ]:
for current in [run]:
    cp.show(sink, dg.plot_parameters(current))
    cp.show(sink, dg.plot_resources(current))
report.add('9. Parameters and resources', '\n\n'.join(dg.summary(c) for c in [run]))

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))